## CT and RT structure preprocessing for multi-label survival prediction using planning CT

### 1. DICOM and RTSTRUCT to NIfTI conversion

In [ ]:
from dcmrtstruct2nii import dcmrtstruct2nii
import glob
import logging
import os

logger = logging.getLogger(__name__)

In [ ]:
# Download the source data from TCIA and set these paths before running the notebook.
oriPath = '/absolute/path/to/RADCURE'
patients = os.listdir(oriPath)

savePath = '/absolute/path/to/OPC-Radiomics-Nii'
if not os.path.exists(savePath):
   os.makedirs(savePath)

In [ ]:
for p in patients:
    if p == 'LICENSE':
        continue  
    if p in os.listdir(savePath):
        logger.info('Already processed: %s', p)
        continue
    folder = os.listdir(os.path.join(oriPath, p))
    if len(folder)==0:
        logger.warning('No data found for %s', p)
        continue
    subFolder = os.listdir(os.path.join(oriPath, p, folder[0]))
    if len(subFolder)<2:
        logger.warning('Incomplete data for %s', p)
        continue
    pSavePath = os.path.join(savePath,p)

    RTfile = None
    imagePath = None

    for s in subFolder:
        if len(os.listdir(os.path.join(oriPath, p, folder[0], s)))==1:
            RTfile = glob.glob(os.path.join(oriPath, p, folder[0], s,'1-1*'))
        else:
            imagePath = os.path.join(oriPath, p, folder[0], s)
    try:
        dcmrtstruct2nii(RTfile[0],imagePath,pSavePath)
        logger.info('Generated image and RTSTRUCT NIfTI files for %s', p)
    except Exception:
        logger.exception('Failed to convert %s', p)


In [ ]:
savedPatients = os.listdir(savePath)
len(savedPatients)

### 2. Figure out how many patient data are available

In [ ]:
missing_gtv = []
missing_img = []
for p in savedPatients:
    path = os.path.join(savePath, p)
    files = os.listdir(path)

    if 'image.nii.gz' in files:
        if any('GTVp' in file and file.endswith('.nii.gz') for file in files):
            continue
        else:
            missing_gtv.append(p)
            logger.warning('Patient %s does not have a GTV mask', p)
            
    else:
        missing_img.append(p)
        logger.warning('Patient %s does not have a CT image', p)

In [ ]:
len(missing_gtv) # actually, for most of these patients, they have data but they don't have gtvp, we can use empty image (0 image) for them

In [ ]:
len(missing_img)

In [ ]:
avaliable_patient = list(set(savedPatients) - set(missing_gtv) - set(missing_img))
len(avaliable_patient)

### 3. Resampling

In [ ]:
import numpy as np
import SimpleITK as sitk

In [ ]:
resample_path = '/absolute/path/to/OPC-Radiomics-Nii-resample'
if not os.path.exists(resample_path):
   os.makedirs(resample_path)

In [ ]:
resampler = sitk.ResampleImageFilter()
resampler.SetOutputDirection([1, 0, 0, 0, 1, 0, 0, 0, 1])
resampling = [2,2,2] # mm
resampler.SetOutputSpacing(resampling)

In [ ]:
def get_bouding_boxes(ct, pt):
    """
    Get the bounding boxes of the CT and PT images.
    This works since all images have the same direction
    """

    ct_origin = np.array(ct.GetOrigin())
    pt_origin = np.array(pt.GetOrigin())

    ct_position_max = ct_origin + np.array(ct.GetSize()) * np.array(
        ct.GetSpacing())
    pt_position_max = pt_origin + np.array(pt.GetSize()) * np.array(
        pt.GetSpacing())
    return np.concatenate(
        [
            np.maximum(ct_origin, pt_origin),
            np.minimum(ct_position_max, pt_position_max),
        ],
        axis=0,
    )

In [ ]:
def resample_one_patient(p):
    ct = sitk.ReadImage(os.path.join(savePath, p, 'image.nii.gz'))
    label = sitk.ReadImage(os.path.join(savePath, p, 'mask_GTVp.nii.gz'))
    bb = get_bouding_boxes(ct, ct)
    size = np.round((bb[3:] - bb[:3]) / resampling).astype(int)
    resampler.SetOutputOrigin(bb[:3])
    resampler.SetSize([int(k) for k in size])
    resampler.SetInterpolator(sitk.sitkBSpline)
    ct = resampler.Execute(ct)
    sitk.WriteImage(ct, os.path.join(resample_path, p+'_image.nii.gz'))
    resampler.SetInterpolator(sitk.sitkNearestNeighbor)
    label = resampler.Execute(label)
    sitk.WriteImage(label, os.path.join(resample_path, p+'_mask_GTV.nii.gz'))

In [ ]:
processed_dir = resample_path
processed_patients = set()
for filename in os.listdir(processed_dir):
    if filename.endswith("_image.nii.gz"):
        patient_id = filename.split("_")[0]  # Extract the patient ID
        processed_patients.add(patient_id)

In [ ]:
for p in avaliable_patient:
    if p not in processed_patients:
        resample_one_patient(p)
    else:
        logger.info('Skipping already processed patient: %s', p)

### 4. Cropping

In [ ]:
def find_centroid(mask):

    stats = sitk.LabelShapeStatisticsImageFilter()
    stats.Execute(mask)
    try:
        centroid_coords = stats.GetCentroid(255)
    except Exception as exc:
        raise ValueError('Unable to calculate the GTV centroid') from exc
    centroid_idx = mask.TransformPhysicalPointToIndex(centroid_coords)

    return np.asarray(centroid_idx, dtype=np.float64)

In [ ]:
crop_path = '/absolute/path/to/OPC-Radiomics-Nii-resample-crop'
if not os.path.exists(crop_path):
   os.makedirs(crop_path)

In [ ]:
for p in avaliable_patient:
    try:
        image_path = os.path.join(resample_path, p + '_image.nii.gz')
        mask_path = os.path.join(resample_path, p + '_mask_GTV.nii.gz')

        image = sitk.ReadImage(image_path)
        mask = sitk.ReadImage(mask_path)

        patch_size = np.array([80, 80, 48])

        tumour_center = find_centroid(mask)
        size = patch_size
        min_coords = np.floor(tumour_center - size / 2).astype(np.int64)
        max_coords = np.floor(tumour_center + size / 2).astype(np.int64)
        min_x, min_y, min_z = min_coords
        max_x, max_y, max_z = max_coords

        if min_x < 0 or min_y < 0 or min_z < 0 or \
           max_x > image.GetSize()[0] or max_y > image.GetSize()[1] or max_z > image.GetSize()[2]:
            logger.warning('Skipping %s: crop indices are out of range', p)
            continue

        image = image[min_x:max_x, min_y:max_y, min_z:max_z]
        mask = mask[min_x:max_x, min_y:max_y, min_z:max_z]

        image = sitk.Clamp(image, sitk.sitkFloat32, -500, 500)

        sitk.WriteImage(image, os.path.join(crop_path, p + '_image.nii.gz'))
        sitk.WriteImage(mask, os.path.join(crop_path, p + '_mask_GTV.nii.gz'))


    except Exception as e:
        logger.exception('Error processing %s: %s', p, e)
        continue

In [ ]:
import csv

with open('avaliable_patient.csv','w') as result_file:
    wr = csv.writer(result_file, dialect='excel')
    for p in avaliable_patient:
        wr.writerow([p])